In [9]:
import torch 
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

In [31]:
class CIFAR10:
    def __init__(self):
        # augmentation
        self.transform = transforms.Compose(
            [transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
        )
        self.trainloader = None 
        self.testloader = None
        self.batch_size = 4
        self.classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

    def generate(self):
        # train dataset
        trainset = torchvision.datasets.CIFAR10(root="./data", train=True, 
                                                download=True, transform=self.transform)
        self.trainloader = torch.utils.data.DataLoader(trainset, batch_size=self.batch_size, 
                                                       shuffle=True, num_workers=2)

        # test dataset
        testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=self.transform)
        self.testloader = torch.utils.data.DataLoader(testset, batch_size=self.batch_size,
                                         shuffle=False, num_workers=2)

cifar_data = CIFAR10() 
cifar_data.generate()

Files already downloaded and verified
Files already downloaded and verified


In [32]:
def imshow(img):
    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

dataiter = iter(cifar_data.trainloader)
images, labels = next(dataiter)

# imshow(images[0])
print("target", cifar_data.classes[labels[0]])
print(images[0].shape)


target truck
torch.Size([3, 32, 32])


In [21]:
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

net = Net()

In [23]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [33]:
# train network
Epochs = 50
for epoch in range(Epochs):
    running_loss = 0.0
    for i, data in enumerate(cifar_data.trainloader, 0):
        inputs, targets = data

        # zero out the gradients
        optimizer.zero_grad()
         
        # forward propogation
        outputs = net(inputs)
        loss = criterion(outputs, targets)
        loss.backward()

        # update weights based on gradients
        optimizer.step()
        running_loss += loss.item()
        # stats every 2000 mini-batches
        if i % 2000 == 1999:
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0

print("finished training!")

[1,  2000] loss: 1.230
[1,  4000] loss: 1.202
[1,  6000] loss: 1.184
[1,  8000] loss: 1.201
[1, 10000] loss: 1.173
[1, 12000] loss: 1.166
[2,  2000] loss: 1.088
[2,  4000] loss: 1.089
[2,  6000] loss: 1.117
[2,  8000] loss: 1.090
[2, 10000] loss: 1.093
[2, 12000] loss: 1.089
[3,  2000] loss: 1.014
[3,  4000] loss: 1.011


KeyboardInterrupt: 

In [30]:
correct = 0
total = 0
with torch.no_grad():
    for data in cifar_data.testloader:
        images, target = data
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')


Accuracy of the network on the 10000 test images: 54 %
